# Hosted Agent Sessions CRUD — `@azure/ai-projects`

This notebook demonstrates create, get, list, and delete operations on Hosted Agent Sessions using the `AIProjectClient`.

Sessions only work with Hosted Agents. It uploads the `basic-agent` code zip as a temporary Hosted Agent version, then creates, retrieves, lists, and deletes a session against that version.

It mirrors the [`sessionsCrud.ts`](./sessionsCrud.ts) sample and runs the **locally built** `@azure/ai-projects` from this repo.

## Prerequisites

1. **Build the package first** so `dist/` is current: `cd sdk/ai/ai-projects && pnpm build`
2. **tslab kernel** installed and registered (`npm install -g tslab` then `tslab install`); select the **TypeScript** (tslab) kernel.
3. **Launch VS Code / Jupyter from `sdk/ai/ai-projects/`** so Node resolves the local `@azure/ai-projects`.
4. **`az login`** completed so `DefaultAzureCredential` can authenticate.
5. **Environment variables**: `FOUNDRY_PROJECT_ENDPOINT`, `FOUNDRY_MODEL_NAME`, `FOUNDRY_HOSTED_AGENT_NAME` (optional; defaults to `MyHostedAgent`).

Run the cells in order (top to bottom); state is shared across cells.

In [1]:
// Imports and configuration
import type {
  CreateAgentVersionFromCodeContent,
  HostedAgentDefinition,
  VersionRefIndicator,
} from "@azure/ai-projects";
import { AIProjectClient } from "@azure/ai-projects";
import { DefaultAzureCredential } from "@azure/identity";
import { createHash } from "node:crypto";
import { readFileSync } from "node:fs";
import path from "node:path";

const projectEndpoint = process.env["FOUNDRY_PROJECT_ENDPOINT"] ?? "<project endpoint>";
const modelName = process.env["FOUNDRY_MODEL_NAME"] ?? "<model deployment name>";
const agentName = process.env["FOUNDRY_HOSTED_AGENT_NAME"] ?? "MyHostedAgent";

const codeZipPath = path.resolve("../assets/basic-agent.zip");

function sha256Hex(data: Uint8Array): string {
  return createHash("sha256").update(data).digest("hex");
}

console.log(`Model: ${modelName}`);
console.log(`Agent: ${agentName}`);

Model: gpt-5.2
Agent: MyTestHostedAgent5
Agent: MyTestHostedAgent5


In [2]:
// Create the AI Project client
const project = new AIProjectClient(projectEndpoint, new DefaultAzureCredential());

In [3]:
// Create a temporary hosted agent version from code
const codeZip = readFileSync(codeZipPath);
const codeZipSha256 = sha256Hex(codeZip);

const definition: HostedAgentDefinition = {
  kind: "hosted",
  cpu: "0.5",
  memory: "1Gi",
  protocol_versions: [{ protocol: "responses", version: "2.0.0" }],
  code_configuration: {
    runtime: "python_3_14",
    entry_point: ["python", "main.py"],
    dependency_resolution: "remote_build",
  },
  environment_variables: {
    FOUNDRY_PROJECT_ENDPOINT: projectEndpoint,
    FOUNDRY_MODEL_NAME: modelName,
  },
};

const content: CreateAgentVersionFromCodeContent = {
  metadata: {
    description: "Sessions CRUD hosted agent uploaded from assets/basic-agent.",
    definition,
  },
  code: { contents: codeZip, contentType: "application/zip", filename: "code.zip" },
};

console.log("Creating hosted agent version from code...");
const created = await project.agents.createVersionFromCode(agentName, codeZipSha256, content);
const createdVersion = created.version;
console.log(`Created code-based hosted agent version: ${createdVersion}`);

Creating hosted agent version from code...
Created code-based hosted agent version: 3


In [4]:
// Poll until the agent version is active
for (let attempt = 0; attempt < 60; attempt++) {
  await new Promise((resolve) => setTimeout(resolve, 3_000));
  const versionDetails = await project.agents.getVersion(agentName, createdVersion);
  const status = versionDetails.status;
  console.log(`Agent version status: ${status} (attempt ${attempt + 1}/60)`);
  if (status === "active") break;
  if (status === "failed") {
    throw new Error(`Agent version provisioning failed: ${JSON.stringify(versionDetails)}`);
  }
  if (attempt === 59) {
    throw new Error("Timed out waiting for agent version to become active");
  }
}

Agent version status: creating (attempt 1/60)
Agent version status: creating (attempt 2/60)
Agent version status: creating (attempt 3/60)
Agent version status: creating (attempt 4/60)
Agent version status: creating (attempt 5/60)
Agent version status: creating (attempt 6/60)
Agent version status: creating (attempt 7/60)
Agent version status: creating (attempt 8/60)
Agent version status: creating (attempt 9/60)
Agent version status: creating (attempt 10/60)
Agent version status: creating (attempt 11/60)
Agent version status: active (attempt 12/60)


In [5]:
// Create a session
const versionIndicator: VersionRefIndicator = {
  type: "version_ref",
  agent_version: createdVersion,
};
const session = await project.agents.createSession(agentName, versionIndicator);
console.log(`Created session (id: ${session.agent_session_id}, status: ${session.status})`);

Created session (id: bd3025ec5bc2d0cb00ULskmglhxqEvTGkfLDdFHumrRPOgHUqD, status: active)


In [6]:
// Retrieve the session by its ID
const fetched = await project.agents.getSession(agentName, session.agent_session_id);
console.log(`Retrieved session (id: ${fetched.agent_session_id}, status: ${fetched.status})`);

Retrieved session (id: bd3025ec5bc2d0cb00ULskmglhxqEvTGkfLDdFHumrRPOgHUqD, status: active)


In [10]:
// List sessions for the agent
console.log("Listing sessions for the agent...");
console.log("Sessions:");
const listSessions = async () => {
  for await (const item of project.agents.listSessions(agentName)) {
    console.log(`  - ${item.agent_session_id} (status: ${item.status})`);
  }
};
await listSessions();

Listing sessions for the agent...
Sessions:
Sessions:
  - 360183f8d7b53662afc59ec65543a9383231917273b638d50bd2275de585994 (status: idle)
  - bd3025ec5bc2d0cb00ULskmglhxqEvTGkfLDdFHumrRPOgHUqD (status: active)


In [11]:
// Delete the session
await project.agents.deleteSession(agentName, session.agent_session_id);
console.log(`Deleted session (id: ${session.agent_session_id})`);

Deleted session (id: bd3025ec5bc2d0cb00ULskmglhxqEvTGkfLDdFHumrRPOgHUqD)


In [12]:
// Clean up the temporary agent version
await project.agents.deleteVersion(agentName, createdVersion, { force: true });
console.log(`Agent version ${createdVersion} deleted.`);

Agent version 3 deleted.
